In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# Plot for George

### GridWorld

Step 1. Look at tuning

We start with:
- `04_23_2024/exp_0.py`: Tune PDA on GridWorld and LunarLander.
- `04_23_2024/exp_1.py`: Tune PPO and DQN on PDA and LunarLander.

In [ ]:
path = "/Users/calebju/Code/RL-general-action-state/logs"

# 04_23_2024/exp_0.py
data_1 = np.zeros(18)
for i in range(len(data_1)):
    df = pd.read_csv(os.path.join(path, "04_23_2024/exp_0/run_%d/seed=0.csv" % i))
    data_1[i] = df['episode rewards'].iloc[-1]

# 04_23_2024/exp_1.py
data_2 = np.zeros(24)
for i in range(len(data_2)):
    df = pd.read_csv(os.path.join(path, "04_23_2024/exp_1/run_%d/seed=0.csv" % i))
    if len(df['episode rewards']) == 0:
        data_2[i] = np.inf
        continue
    data_2[i] = df['episode rewards'].iloc[-1]

First we look at `GridWorld-v0`.

In [ ]:
print("Experiment 04_23_2024/exp_0.py GridWorld\n")
print("id|    score\n------------")
for i in range(9):
    print("%d |%.2e %s" % (i, -data_1[i], "*" if -data_1[i] == np.min(-data_1[:9]) else ""))

This corresponds to:
- pmd-linear-kl with `lr=1.0`
- pmd-nn-kl with `lr=0.1`
- pmd-nn-tsallis with `lr=0.1`


Next we look at `LunarLander-v3`.

In [ ]:
print("Experiment 04_23_2024/exp_0.py LunarLander\n")
print("id|    score\n------------")
for i in range(9,18):
    print("%d |%.2e %s" % (i, -data_1[i], "*" if -data_1[i] == np.min(-data_1[9:18]) else ""))

This corresponds to 
- pmd-linear-kl with `lr=0.1`
- pmd-nn-kl with `lr=0.1`
- pmd-nn-tsallis with `lr=0.1`

#### Tune PPO and DQN

Next we look at PPO for `GridWorld-v0`

In [ ]:
print("Experiment 04_23_2024/exp_1.py GridWorld-v0 PPO\n")
print("id|    score\n------------")
for i in range(0,6):
    print("%d |%.2e %s" % (i, -data_2[i], "*" if -data_2[i] == np.min(-data_2[0:6]) else ""))

This corresponds to either `lr=0.001`.

Now DQN on `GridWorld-v0`.

In [ ]:
print("Experiment 04_23_2024/exp_1.py GridWorld-v0 PPO\n")
print("id|    score\n------------")
for i in range(6,12):
    print("%d |%.2e %s" % (i, -data_2[i], "*" if -data_2[i] == np.min(-data_2[6:10]) else ""))

This corresponds to `lr=0.0003`.

Now PPO on `LunarLander-v3`.

In [ ]:
print("Experiment 04_23_2024/exp_1.py GridWorld-v0 PPO\n")
print("id|    score\n------------")
for i in range(12,18):
    print("%d |%.2e %s" % (i, -data_2[i], "*" if -data_2[i] == np.min(-data_2[12:18]) else ""))

This corrsponds to `lr=0.001`.

Finally DQN on `LunarLander-v3`.

In [ ]:
print("Experiment 04_23_2024/exp_1.py GridWorld-v0 PPO\n")
print("id|    score\n------------")
for i in range(18,24):
    print("%d |%.2e %s" % (i, -data_2[i], "*" if -data_2[i] == np.min(-data_2[18:24]) else ""))

This corresponds to `lr=0.0003`.

Nowe we plot the performance, starting with GridWorld-v0 (George only asked for GridWorld).

In [ ]:
path = "/Users/calebju/Code/RL-general-action-state/logs"

# 02_25_2026/exp_0.py
n_seeds = 10
n_iters = 100_000
data_1 = np.zeros((5,n_seeds,n_iters))
aux_1 = np.zeros((5,n_seeds,n_iters), dtype=float)
len_1 = np.zeros((5, n_seeds), dtype=int)
for i in range(data_1.shape[0]):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(path, "02_25_2026/exp_0/run_%d/seed=%d.csv" % (i,j)))
        temp = df['episode rewards']
        temp_ = df['episode len']
        len_1[i,j] = len(temp)
        data_1[i,j,:len_1[i,j]] = temp
        aux_1[i,j,:len_1[i,j]] = np.cumsum(temp_)

Clean it.

In [ ]:
xs = np.append(
    np.append(np.arange(1_000, step=10), np.arange(1_000, 10_000, step=25)), 
    np.arange(10_000, 95_000, step=100)
)
clean_1 = np.zeros((data_1.shape[0], n_seeds, len(xs)), dtype=int)
for i in range(clean_1.shape[0]):
    for j in range(n_seeds):
        for k,x in enumerate(xs):
            # finds index i such that len_i < x <= len_(i+1)
            clean_1[i,j,k] = np.argmax(x <= aux_1[i,j,:])
            if clean_1[i,j,k] < clean_1[i,j,k-1]:
                print("Wrong index at (i,j,k)=(%d,%d,%d) | x=%d max=%d" % (i,j,k,x,np.max(aux_1[i,j,:])))

Now we plot the performance for GridWorld.

In [ ]:
plt.style.use('default')
_, ax = plt.subplots(figsize=(5,4))
label_arr = ['PDA (linear-kl)', 'PDA (nn-kl)', 'PDA (nn-tsallis)', 'PPO', 'DQN']
color_arr = ['#1f77b4', 'blue', 'teal', '#ff7f0e', '#8c564b']
ones_5 = 0.2*np.ones(5)
offs = len(ones_5)-1

aa = [1,2,0,3,4]
for i in aa:
    ys = np.zeros(clean_1.shape[1:])
    for j in range(n_seeds):
        ys[j] = data_1[i, j, clean_1[i,j]]
        ys[j] = np.convolve(ys[j], ones_5)[:-len(ones_5)+1]
    med = np.mean(ys, axis=0)
    rng = np.std(ys, axis=0)
    rng *= 2.571 # based on 2-sided t-score with p=0.05
    ax.plot(xs[offs:], med[offs:], label=label_arr[i], color=color_arr[i], linestyle="solid" if i > 0 else "dotted")
    ax.fill_between(xs[offs:], (med-rng)[offs:], (med+rng)[offs:], color=color_arr[i], alpha=0.1)

ax.legend(loc="upper left")
ax.set(
    title="GridWorld-v0",
    ylabel="Discounted episodic reward\n smoothed over %d periods" % len(ones_5),
    xlabel="Env Steps",
    ylim=(-250,150),
    xlim=(-500, 95_000),
)

plt.tight_layout()
# plt.savefig("humanoid.png", dpi=90)
plt.savefig("gridworld_jg_style.png", dpi=240)

We can also add a head-to-head.

In [ ]:
plt.style.use('default')
_, axes = plt.subplots(ncols=2, figsize=(8,4))
ones_5 = 0.2*np.ones(5)
offs = len(ones_5)-1

aa = [1,3]
for k,i in enumerate(aa):
    for j in range(n_seeds):
        ys = np.convolve(data_1[i, j, clean_1[i,j]], ones_5)[:-len(ones_5)+1]
        axes[k].plot(xs[offs:], ys[offs:])

axes[0].set(
    title="PDA on GridWorld-v0",
    ylabel="Discounted episodic reward\n smoothed over %d periods" % len(ones_5),
    xlabel="Env Steps",
    ylim=(-250,150),
    xlim=(-500, 95_000),
)
axes[1].set(
    title="PPO on GridWorld-v0",
    xlabel="Env Steps",
    ylim=(-250,150),
    xlim=(-500, 95_000),
)

plt.tight_layout()
# plt.savefig("humanoid.png", dpi=90)
plt.savefig("gridworld_seed2seed_jg_style.png", dpi=240)

### Humanoid

Step 1. Get the data

In [ ]:
path = "/Users/calebju/Code/RL-general-action-state/logs"
n_seeds = 5

# 02_12_2026/exp_0.py 
data_2 = np.zeros((4,n_seeds,500_000), dtype=float)
aux_2 = np.zeros((4,n_seeds,500_000), dtype=float)
len_2 = np.zeros((4, n_seeds), dtype=int)
for i in range(len(data_2)):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(path, "02_12_2026/exp_1/run_%d/seed=%d.csv" % (i, j)))
        temp = df['episode rewards']
        temp2 = df['episode len']
        len_2[i,j] = len(temp)
        # data_1[i,j,:len_1[i,j]] = np.convolve(temp, ones_5)[:-2]
        data_2[i,j,:len_2[i,j]] = temp
        aux_2[i,j,:len_2[i,j]] = np.cumsum(temp2)

Step 2. Bucket the data

In [ ]:
xs = np.append(
    np.append(np.arange(1_000, step=10), np.arange(1_000, 10_000, step=25)), 
    np.arange(10_000, 100_000, step=100)
)
clean_2 = np.zeros((data_2.shape[0], n_seeds, len(xs)), dtype=int)
for i in range(clean_2.shape[0]):
    for j in range(clean_2.shape[1]):
        for k,x in enumerate(xs):
            # finds index i such that len_i < x <= len_(i+1)
            clean_2[i,j,k] = np.argmax(x <= aux_2[i,j,:])

Step 3. Plot the data

In [ ]:
plt.style.use('default')
_, ax = plt.subplots(figsize=(5,4))
label_arr = ['PDA', 'PPO', 'DDPG', 'PPO (Zoo)']
color_arr = ['#1f77b4', '#ff7f0e', '#8c564b', 'yellow']
ones_5 = 0.05*np.ones(20)

aa = [0,1,3,2]
for i in aa:
    ys = np.zeros(clean_2.shape[1:])
    for j in range(n_seeds):
        ys[j] = data_2[i, j, clean_2[i,j]]
        ys[j] = np.convolve(ys[j], ones_5)[:-len(ones_5)+1]
    med = np.mean(ys, axis=0)
    rng = np.std(ys, axis=0)
    rng *= 2.571 # based on 2-sided t-score with p=0.05
    ax.plot(xs, med, label=label_arr[i], color=color_arr[i])
    ax.fill_between(xs, med-rng, med+rng, color=color_arr[i], alpha=0.1)

ax.legend(loc="upper left")
ax.set(
    title="Humanoid-v5",
    ylabel="Discounted episodic reward\n smoothed over %d periods" % len(ones_5),
    xlabel="Env Steps",
    ylim=(0,525),
    xlim=(-2_000, 100_000),
)

plt.tight_layout()
# plt.savefig("humanoid.png", dpi=90)
plt.savefig("humanoid_jg_style.png", dpi=240)

In [ ]:
plt.style.use('default')
_, axes = plt.subplots(ncols=2, figsize=(6,4))

i_s = [0,2]
ones_5 = np.ones(100)
ones_5 /= len(ones_5)

for k,i in enumerate(i_s):
    ys = np.zeros(aux_2.shape[1:])
    for j in range(n_seeds):
        l = len_2[i,j]
        _xs = aux_2[i,j,:l:]
        ys = np.convolve(data_2[i,j,:l:], ones_5)[:-len(ones_5)+1]
        axes[k].plot(_xs, ys, label="seed %d" % (j+1))

    # ax.legend(loc="right")
    axes[k].set(
        ylim=(0,525),
        xlim=(-2_000, 100_000),
        xlabel="Samples",
    )
    
axes[0].legend(loc="upper right")
axes[0].set(
    ylabel="Discounted episodic reward\n smoothed over %d periods" % len(ones_5),
    xlabel="Samples",
    ylim=(0,375),
    xlim=(-2_000, 100_000),
)
axes[1].tick_params(labelleft=False)   

plt.suptitle("Humanoid-v5 seed-to-seed comparison (PDA left, DDPG right)")

plt.tight_layout()
# plt.savefig("humanoid_seeds.png", dpi=90)
plt.savefig("humanoid_seeds_jg_style.png", dpi=120)